In [ ]:
# ==========================================
# CÉLULA 1: Importações e Configuração
# ==========================================
# Objetivo: Carregar as bibliotecas necessárias e configurar os parâmetros 
# de comunicação com a API Entrez do NCBI (PubMed Central).
# ==========================================

# Bibliotecas padrão do Python para sistema e tempo
import os
import time

# Expressões regulares para extração de padrões textuais
import re

# Bibliotecas de requisição e parsing web
import requests
from bs4 import BeautifulSoup

# Bibliotecas de manipulação de dados
import pandas as pd
import numpy as np

# ------------------------------------------
# Configurações Globais da API Entrez (NCBI)
# ------------------------------------------

# Base URLs para os utilitários E-utilities
ENTREZ_ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
ENTREZ_EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

# Headers de identificação (Boas práticas exigidas pelo NCBI)
HEADERS = {
    "User-Agent": "UFPB_DataScience_Bot/1.0 (mailto:estudante@ufpb.br)",
    "Accept": "application/xml, application/json"
}

# Parâmetros de resiliência de rede
TIMEOUT = 15.0       # Tempo máximo de espera por requisição (em segundos)
DELAY = 0.4          # Atraso entre requisições para evitar bloqueio (NCBI permite até 3 req/seg sem API key)

# Criação do diretório para armazenar os XMLs brutos localmente
XML_DIR = "PMC_XML"
if not os.path.exists(XML_DIR):
    os.makedirs(XML_DIR)
    print(f"[INFO] Diretório '{XML_DIR}' criado para armazenamento dos artigos.")
else:
    print(f"[INFO] Diretório '{XML_DIR}' já existe. Pronto para uso.")

print("[SUCCESS] Bibliotecas importadas e ambiente Entrez configurado com sucesso!")

[INFO] Diretório 'PMC_XML' já existe. Pronto para uso.
[SUCCESS] Bibliotecas importadas e ambiente Entrez configurado com sucesso!


In [12]:
# ==========================================
# CÉLULA 2: Configuração Global e Dicionários
# ==========================================
# Objetivo: Centralizar a QUERY de busca no PMC, definir limite de retorno (RETMAX),
# estruturar a lista de autores de interesse, a blacklist de substâncias (controles)
# e os dicionários léxicos (palavras-chave) para a mineração de texto (NLP rules).
# ==========================================

# 1. PARÂMETROS DE BUSCA NO PUBMED CENTRAL
# ------------------------------------------
RETMAX = 2000  # Limite máximo de PMCIDs para recuperar

# Query estruturada otimizada para abranger artigos experimentais no modelo PTZ em camundongos
QUERY = (
    "(pentylenetetrazole[Text Word] OR PTZ[Text Word]) "
    "AND (seizures[Text Word] OR convulsion[Text Word] OR convulsions[Text Word]) "
    "AND (mice[Text Word] OR mouse[Text Word]) "
    "AND (anticonvulsant[Text Word] OR anticonvulsants[Text Word] OR neuroprotection[Text Word] "
    "OR neuroprotective[Text Word] OR antiepileptic[Text Word] OR antiepileptics[Text Word]) "
    "AND (experiment* OR treatment OR administration OR dose OR mg/kg) "
    "AND (\"2011/01/01\"[Date - Publication] : \"2026/12/31\"[Date - Publication]) "
    "NOT Review[Publication Type]"
)

# 2. AUTORES DE INTERESSE
# ------------------------------------------
# Nota: A presença na lista não é obrigatória para inclusão, mas auxilia na 
# identificação de estudos produzidos por grupos de pesquisa de referência.
AUTHORS = [
    "Reinaldo Nobrega de Almeida",
    "Reinaldo Nóbrega de Almeida",
    "Almeida RN",
    "Cicero Francisco Bezerra Felipe",
    "Cícero Francisco Bezerra Felipe",
    "Felipe CFB",
    "Marta Regina Kerntopf",
    "Kerntopf MR",
    "Alefe Brito Monteiro",
    "Álefe Brito Monteiro",
    "Monteiro AB"
]

# 3. BLACKLIST DE SUBSTÂNCIAS
# ------------------------------------------
# Ignoraremos essas substâncias na etapa de identificação do "Composto Experimental"
# pois representam controles negativos (veículos) ou positivos (fármacos padrão).
BLACKLIST = [
    "control", "vehicle", "saline", "tween", "ptz alone", 
    "diazepam", "dzp", "clonazepam", "phenytoin", "valproate", 
    "valproic acid", "phenobarbital", "carbamazepine", 
    "positive control", "negative control"
]

# 4. DICIONÁRIO DE PALAVRAS-CHAVE (LÉXICO EXPERIMENTAL)
# ------------------------------------------
# Mapeamento para extração baseada em regras contextuais e Regex.
KEYWORDS = {
    "primeira_crise": [
        "latency to first seizure",
        "latency of first seizure",
        "latency to first convulsion",
        "latency of first convulsion",
        "latency to tonic-clonic seizure",
        "first myoclonic jerk"
    ],
    "morte": [
        "latency to death",
        "death latency",
        "time to death",
        "time for death"
    ],
    "significancia": [
        "p < 0.05",
        "p<0.05",
        "significantly",
        "statistically significant",
        "*",
        "**"
    ],
    "efeito_positivo": [
        "increased latency",
        "prolonged latency",
        "delayed onset",
        "prevented seizure",
        "protected",
        "anticonvulsant activity",
        "neuroprotective effect"
    ],
    "efeito_negativo": [
        "no effect",
        "ineffective",
        "failed to protect",
        "no anticonvulsant effect"
    ]
}

print("[INFO] Configurações globais e dicionários léxicos carregados em memória.")
print(f"[INFO] Limite de recuperação definido para: {RETMAX} artigos.")

[INFO] Configurações globais e dicionários léxicos carregados em memória.
[INFO] Limite de recuperação definido para: 2000 artigos.


In [13]:
# ==========================================
# CÉLULA 3: Busca dos PMCIDs
# ==========================================
# Objetivo: Executar a query estruturada no PubMed Central (via esearch) 
# e recuperar a lista de identificadores únicos (PMCIDs) dos artigos.
# ==========================================

print("[INFO] Iniciando busca no PubMed Central (PMC)...")

# Parâmetros da requisição esearch
esearch_params = {
    "db": "pmc",
    "term": QUERY,
    "retmax": RETMAX,
    "retmode": "json",
    "usehistory": "n"
}

pmc_ids = []

try:
    # Requisição GET para a API Entrez
    response = requests.get(
        ENTREZ_ESEARCH_URL, 
        params=esearch_params, 
        headers=HEADERS, 
        timeout=TIMEOUT
    )
    
    # Levanta uma exceção caso o status HTTP não seja 200 (OK)
    response.raise_for_status()
    
    # Parse do JSON de resposta
    data = response.json()
    
    # Extração defensiva dos dados
    if "esearchresult" in data and "idlist" in data["esearchresult"]:
        pmc_ids = data["esearchresult"]["idlist"]
        total_found = int(data["esearchresult"].get("count", 0))
        
        print(f"[SUCCESS] Busca concluída com sucesso!")
        print(f" - Total de artigos encontrados no PMC: {total_found}")
        print(f" - Quantidade de IDs recuperados: {len(pmc_ids)}")
        
        if len(pmc_ids) > 0:
            print(f" - Primeiros PMCIDs recuperados: {pmc_ids[:5]}")
    else:
        print("[ERROR] Estrutura JSON inesperada. A chave 'idlist' não foi encontrada.")
        print(f"Resposta bruta da API: {data}")

except requests.exceptions.Timeout:
    print("[ERROR] Timeout na requisição. O servidor do NCBI não respondeu a tempo.")
except requests.exceptions.HTTPError as http_err:
    print(f"[ERROR] Erro HTTP retornado pelo servidor: {http_err}")
except requests.exceptions.RequestException as e:
    print(f"[ERROR] Falha genérica na comunicação com a API: {e}")
except ValueError:
    print("[ERROR] Falha ao decodificar a resposta JSON do servidor.")

# Aplicando o delay de segurança exigido pelas políticas do NCBI (máx 3 requisições/seg)
time.sleep(DELAY)

[INFO] Iniciando busca no PubMed Central (PMC)...
[SUCCESS] Busca concluída com sucesso!
 - Total de artigos encontrados no PMC: 435
 - Quantidade de IDs recuperados: 435
 - Primeiros PMCIDs recuperados: ['13336032', '13275497', '13149591', '13137726', '12014963']


In [14]:
# ==========================================
# CÉLULA 4: Download dos XMLs Completos
# ==========================================
# Objetivo: Baixar o texto completo em formato XML para cada PMCID recuperado,
# armazenando-os tanto em memória (dicionário) quanto em arquivos locais (.xml).
# ==========================================

print(f"[INFO] Preparando download de {len(pmc_ids)} artigos do PubMed Central...")

# Dicionário na memória para armazenar o XML bruto indexado pelo PMCID
xml_articles = {}

success_count = 0
fail_count = 0

# Iteração controlada sobre a lista de IDs obtida na Célula 3
for index, pmcid in enumerate(pmc_ids, 1):
    # Opcional: PMCIDs podem vir prefixados com "PMC" ou apenas o número. Padronizamos:
    clean_id = pmcid.replace("PMC", "")
    filename = os.path.join(XML_DIR, f"PMC{clean_id}.xml")
    
    # Flag para controlar se precisamos fazer a requisição HTTP ou ler do disco
    xml_content = None
    
    # 1. VERIFICAÇÃO DE CACHE LOCAL (Evita re-fazer download se o arquivo já existir)
    if os.path.exists(filename):
        try:
            with open(filename, "r", encoding="utf-8") as f:
                xml_content = f.read()
            xml_articles[f"PMC{clean_id}"] = xml_content
            success_count += 1
            continue  # Pula para o próximo ID sem gastar tempo de rede
        except Exception as e:
            print(f"[WARN] Erro ao ler arquivo local PMC{clean_id}.xml: {e}. Tentando download...")

    # 2. DOWNLOAD VIA API EFETCH
    efetch_params = {
        "db": "pmc",
        "id": clean_id,
        "rettype": "full",
        "retmode": "xml"
    }
    
    try:
        response = requests.get(
            ENTREZ_EFETCH_URL, 
            params=efetch_params, 
            headers=HEADERS, 
            timeout=TIMEOUT
        )
        
        if response.status_code == 200 and "</pmc-articleset>" in response.text:
            xml_content = response.text
            
            # Armazena na estrutura em memória
            xml_articles[f"PMC{clean_id}"] = xml_content
            
            # Persiste no disco para cache futuro
            with open(filename, "w", encoding="utf-8") as f:
                f.write(xml_content)
                
            success_count += 1
        else:
            print(f"[WARN] Falha na validação do XML para o ID PMC{clean_id}. Status: {response.status_code}")
            fail_count += 1
            
    except Exception as e:
        print(f"[WARN] Erro ao baixar o artigo PMC{clean_id}: {e}")
        fail_count += 1
        
    # Log de progresso a cada 25 artigos
    if index % 25 == 0 or index == len(pmc_ids):
        print(f" -> Progresso: {index}/{len(pmc_ids)} processados (Sucessos: {success_count}, Falhas: {fail_count})")
        
    # Respeito estrito ao Rate Limiting da API do NCBI
    time.sleep(DELAY)

print("\n[SUCCESS] Fase de aquisição de dados concluída!")
print(f" - Total de artigos em memória/disco: {len(xml_articles)}")
print(f" - Sucessos no ciclo atual: {success_count}")
print(f" - Falhas no ciclo atual: {fail_count}")

[INFO] Preparando download de 435 artigos do PubMed Central...
 -> Progresso: 375/435 processados (Sucessos: 375, Falhas: 0)

[SUCCESS] Fase de aquisição de dados concluída!
 - Total de artigos em memória/disco: 435
 - Sucessos no ciclo atual: 435
 - Falhas no ciclo atual: 0


In [15]:
# ==========================================
# CÉLULA 5: Parser Inteligente do XML
# ==========================================
# Objetivo: Estruturar o texto bruto do XML JATS em campos categóricos úteis 
# para a mineração (Title, Abstract, Methods, Results, Discussion, etc.),
# lidando com as inconsistências de nomenclatura de seções das revistas.
# ==========================================

print("[INFO] Iniciando o parsing estruturado dos documentos XML...")

from bs4 import BeautifulSoup
import re

# Dicionário final estruturado
articles_xml = {}

for pmcid, xml_content in xml_articles.items():
    # Inicializando estrutura vazia para o artigo
    article_data = {
        "title": "",
        "authors": [],
        "abstract": "",
        "methods": "",
        "results": "",
        "discussion": "",
        "conclusion": "",
        "tables": [],
        "figures": []
    }
    
    try:
        # Usando lxml-xml para parsing rápido e correto de tags XML
        soup = BeautifulSoup(xml_content, "lxml-xml")
        
        # 1. Título do Artigo
        title_node = soup.find("article-title")
        if title_node:
            article_data["title"] = title_node.get_text(strip=True, separator=" ")
            
        # 2. Autores
        # No JATS, autores estão em <contrib contrib-type="author">
        for author_node in soup.find_all("contrib", {"contrib-type": "author"}):
            surname = author_node.find("surname")
            given = author_node.find("given-names")
            if surname and given:
                full_name = f"{surname.get_text(strip=True)} {given.get_text(strip=True)}"
                # Formato alternativo (Sobrenome Iniciais)
                initials = "".join([n[0] for n in given.get_text(strip=True).split() if n])
                alt_name = f"{surname.get_text(strip=True)} {initials}"
                
                article_data["authors"].extend([full_name, alt_name])
                
        # 3. Abstract
        abstract_node = soup.find("abstract")
        if abstract_node:
            article_data["abstract"] = abstract_node.get_text(strip=True, separator=" ")
            
        # 4. Parsing Inteligente de Seções do Corpo (Body)
        # Varremos todas as tags <sec> buscando títulos para mapeamento
        for sec in soup.find_all("sec"):
            title_tag = sec.find("title")
            if not title_tag:
                continue
                
            sec_title = title_tag.get_text(strip=True).lower()
            sec_text = sec.get_text(strip=True, separator=" ")
            
            # Classificação heurística das seções
            if re.search(r"method|material|experiment|protocol", sec_title):
                article_data["methods"] += sec_text + " "
            elif re.search(r"result", sec_title):
                article_data["results"] += sec_text + " "
            elif re.search(r"discussion", sec_title):
                article_data["discussion"] += sec_text + " "
            elif re.search(r"conclusion", sec_title):
                article_data["conclusion"] += sec_text + " "

        # 5. Tabelas e Legendas de Figuras
        # Extrair texto interno das tabelas para futura mineração de doses/latências
        for table in soup.find_all("table-wrap"):
            article_data["tables"].append(table.get_text(strip=True, separator=" "))
            
        for fig in soup.find_all("fig"):
            # O texto da tag <fig> geralmente contém o caption que pode relatar p-values
            article_data["figures"].append(fig.get_text(strip=True, separator=" "))
            
        # Salvando o artigo processado
        articles_xml[pmcid] = article_data
        
    except Exception as e:
        print(f"[WARN] Falha ao processar a estrutura do artigo {pmcid}: {e}")

print(f"[SUCCESS] Parsing concluído! Artigos processados: {len(articles_xml)}")

# ==========================================
# DIAGNÓSTICO E VALIDAÇÃO DA CÉLULA
# ==========================================
if articles_xml:
    sample_pmcid = list(articles_xml.keys())[0]
    sample_data = articles_xml[sample_pmcid]
    
    print("\n--- DIAGNÓSTICO DO PRIMEIRO ARTIGO (AMOSTRA) ---")
    print(f"PMCID: {sample_pmcid}")
    print(f"Título: {sample_data['title'][:150]}...")
    print(f"Total de Autores mapeados (variações inclusas): {len(sample_data['authors'])}")
    print(f"Comprimento do Abstract: {len(sample_data['abstract'])} caracteres")
    print(f"Comprimento de Methods: {len(sample_data['methods'])} caracteres")
    print(f"Comprimento de Results: {len(sample_data['results'])} caracteres")
    print(f"Comprimento de Discussion: {len(sample_data['discussion'])} caracteres")
    print(f"Comprimento de Conclusion: {len(sample_data['conclusion'])} caracteres")
    print(f"Quantidade de Tabelas: {len(sample_data['tables'])}")
    print(f"Quantidade de Figuras: {len(sample_data['figures'])}")
    print("------------------------------------------------")

[INFO] Iniciando o parsing estruturado dos documentos XML...
[SUCCESS] Parsing concluído! Artigos processados: 435

--- DIAGNÓSTICO DO PRIMEIRO ARTIGO (AMOSTRA) ---
PMCID: PMC13336032
Título: Enhanced Glycolysis‐Driven Histone H3K18 Lactylation Regulates Epileptogenesis by Modulating the E3 Ubiquitin Ligase COP1...
Total de Autores mapeados (variações inclusas): 18
Comprimento do Abstract: 1299 caracteres
Comprimento de Methods: 23975 caracteres
Comprimento de Results: 52494 caracteres
Comprimento de Discussion: 14227 caracteres
Comprimento de Conclusion: 1108 caracteres
Quantidade de Tabelas: 0
Quantidade de Figuras: 7
------------------------------------------------


In [16]:
# ==========================================
# CÉLULA 6: Validação Científica dos Artigos
# ==========================================
# Objetivo: Filtrar os artigos extraídos garantindo que sejam estudos
# primários, experimentais, utilizando o modelo PTZ em camundongos,
# com aferição quantitativa de latência/morte e análise estatística.
# ==========================================

print("[INFO] Iniciando validação científica do corpus...")

import re

# Dicionário para armazenar apenas os artigos que passarem nos testes
validated_articles = {}

# Contadores para o relatório de exclusão
exclusion_stats = {
    "Texto vazio ou ausência de seções experimentais": 0,
    "Modelo animal divergente (sem camundongo)": 0,
    "Modelo PTZ não detectado": 0,
    "Ausência de administração de doses (mg/kg)": 0,
    "Sem métricas de latência para crise/morte": 0,
    "Ausência de descritores estatísticos (Média/SD/SEM/p-value)": 0
}

for pmcid, data in articles_xml.items():
    # Unificamos o texto das seções relevantes para facilitar a busca heurística.
    # Convertendo tudo para minúsculas para padronizar o regex (case-insensitive flag também serve, mas assim otimizamos).
    full_search_text = " ".join([
        data.get("abstract", ""),
        data.get("methods", ""),
        data.get("results", ""),
        data.get("discussion", ""),
        " ".join(data.get("tables", [])),
        " ".join(data.get("figures", []))
    ]).lower()
    
    # REGRA 1: Filtro de densidade informacional (Remove resumos vazios ou revisões curtas)
    if len(full_search_text.strip()) < 1000:
        exclusion_stats["Texto vazio ou ausência de seções experimentais"] += 1
        continue
        
    # REGRA 2: Animal (mouse / mice)
    if not re.search(r"\b(mice|mouse)\b", full_search_text):
        exclusion_stats["Modelo animal divergente (sem camundongo)"] += 1
        continue
        
    # REGRA 3: Agente convulsivante (PTZ)
    if not re.search(r"\b(ptz|pentylenetetrazole?)\b", full_search_text):
        exclusion_stats["Modelo PTZ não detectado"] += 1
        continue
        
    # REGRA 4: Tratamento Farmacológico (Dose / mg/kg)
    if not re.search(r"(mg/kg|dose\b|doses\b|administration|treatment)", full_search_text):
        exclusion_stats["Ausência de administração de doses (mg/kg)"] += 1
        continue
        
    # REGRA 5: Desfecho esperado (Latência para primeira crise ou morte)
    if not re.search(r"(latency|onset|time to|first seizure|myoclonic|death)", full_search_text):
        exclusion_stats["Sem métricas de latência para crise/morte"] += 1
        continue
        
    # REGRA 6: Rigor Quantitativo (Estatística e dispersão)
    # Procuramos indicadores de significância, média, desvio padrão ou erro padrão
    if not re.search(r"(mean|sd|sem|p\s*<\s*0\.|significant|±)", full_search_text):
        exclusion_stats["Ausência de descritores estatísticos (Média/SD/SEM/p-value)"] += 1
        continue
        
    # Se sobreviveu a todas as regras, o artigo é promovido à próxima fase!
    validated_articles[pmcid] = data

# ==========================================
# RELATÓRIO DE TRIAGEM CIENTÍFICA
# ==========================================
print("\n--- RELATÓRIO DE VALIDAÇÃO ---")
print(f"Total de artigos iniciais: {len(articles_xml)}")
print(f"Artigos APROVADOS: {len(validated_articles)}")
print(f"Artigos EXCLUÍDOS: {len(articles_xml) - len(validated_articles)}")

print("\n--- MOTIVOS DA EXCLUSÃO ---")
for motivo, quantidade in exclusion_stats.items():
    print(f" - {motivo}: {quantidade}")
print("------------------------------")

# Flag para verificar presença dos autores de interesse (Apenas Informativo)
authors_found = 0
for pmcid, data in validated_articles.items():
    article_authors = " ".join(data["authors"]).lower()
    if any(target_author.lower() in article_authors for target_author in AUTHORS):
        authors_found += 1
        
print(f"\n[INFO] Dos {len(validated_articles)} artigos aprovados, {authors_found} contêm pelo menos um autor da sua lista de interesse.")

[INFO] Iniciando validação científica do corpus...

--- RELATÓRIO DE VALIDAÇÃO ---
Total de artigos iniciais: 435
Artigos APROVADOS: 269
Artigos EXCLUÍDOS: 166

--- MOTIVOS DA EXCLUSÃO ---
 - Texto vazio ou ausência de seções experimentais: 28
 - Modelo animal divergente (sem camundongo): 37
 - Modelo PTZ não detectado: 43
 - Ausência de administração de doses (mg/kg): 6
 - Sem métricas de latência para crise/morte: 49
 - Ausência de descritores estatísticos (Média/SD/SEM/p-value): 3
------------------------------

[INFO] Dos 269 artigos aprovados, 1 contêm pelo menos um autor da sua lista de interesse.


In [17]:
# ==========================================
# CÉLULA 7: Identificação da Substância Teste
# ==========================================
# Objetivo: Identificar o composto experimental/extrato avaliado no artigo.
# Utiliza padrões heurísticos (Regex) no Título e Abstract, cruzando 
# os achados com a BLACKLIST para ignorar veículos e fármacos padrão.
# ==========================================

print("[INFO] Iniciando extração das substâncias teste...")

import re

# Padrões sintáticos comuns na literatura farmacológica (case-insensitive)
# O grupo de captura (.*?) tenta isolar o nome da substância.
COMPOUND_PATTERNS = [
    r"(?:anticonvulsant|neuroprotective|antiepileptic)\s+(?:effect|activity|properties|potential|action)s?\s+of\s+([A-Za-z0-9\-\s\']+?)\s+(?:in|on|against)",
    r"(?:effects?|evaluation)\s+of\s+([A-Za-z0-9\-\s\']+?)\s+(?:on|in|against)",
    r"treatment\s+with\s+([A-Za-z0-9\-\s\']+?)\s+(?:significantly|attenuates|protects|reduced|prevented)",
    r"administration\s+of\s+([A-Za-z0-9\-\s\']+?)\s+(?:significantly|attenuates|protects)"
]

extracted_compounds_count = 0

for pmcid, data in validated_articles.items():
    title = data.get("title", "")
    abstract = data.get("abstract", "")
    
    # Texto de busca: Prioridade para o título, depois abstract
    search_text = f"{title}. {abstract}"
    
    found_compound = "Compound Not Identified"
    
    for pattern in COMPOUND_PATTERNS:
        matches = re.finditer(pattern, search_text, re.IGNORECASE)
        
        for match in matches:
            candidate = match.group(1).strip().lower()
            
            # Limpeza rápida de stopwords acidentais na captura
            candidate = re.sub(r"^(the|a|an)\s+", "", candidate)
            
            # Regras de rejeição do candidato:
            # 1. Menor que 3 letras
            # 2. Maior que 50 caracteres (provavelmente capturou uma frase inteira por erro de regex)
            # 3. Está na nossa BLACKLIST exata ou contida no nome (ex: "diazepam treatment")
            if len(candidate) < 3 or len(candidate) > 50:
                continue
                
            is_blacklisted = any(bad_word in candidate for bad_word in BLACKLIST)
            
            if not is_blacklisted:
                found_compound = candidate.title() # Capitaliza a primeira letra de cada palavra
                break # Encontrou um válido, para a busca para este padrão
                
        if found_compound != "Compound Not Identified":
            break # Encontrou o composto, passa para o próximo artigo
            
    # Salva o resultado no dicionário do artigo
    validated_articles[pmcid]["compound"] = found_compound
    
    if found_compound != "Compound Not Identified":
        extracted_compounds_count += 1

print(f"[SUCCESS] Identificação de substâncias concluída!")
print(f" - Compostos identificados com sucesso: {extracted_compounds_count} de {len(validated_articles)} artigos aprovados.")
print(f" - Taxa de sucesso da heurística: {(extracted_compounds_count/len(validated_articles))*100:.2f}%\n")

# Mostrando uma pequena amostra das substâncias extraídas para verificação visual
print("--- AMOSTRA DE COMPOSTOS EXTRAÍDOS ---")
sample_extracted = {k: v["compound"] for k, v in list(validated_articles.items())[:15]}
for p_id, comp in sample_extracted.items():
    print(f"{p_id}: {comp}")
print("--------------------------------------")

[INFO] Iniciando extração das substâncias teste...
[SUCCESS] Identificação de substâncias concluída!
 - Compostos identificados com sucesso: 107 de 269 artigos aprovados.
 - Taxa de sucesso da heurística: 39.78%

--- AMOSTRA DE COMPOSTOS EXTRAÍDOS ---
PMC13336032: Compound Not Identified
PMC13149591: Compound Not Identified
PMC13058834: Compound Not Identified
PMC12990634: Compound Not Identified
PMC12971609: Compound Not Identified
PMC12956243: Compound Not Identified
PMC12954166: Compound Not Identified
PMC12858582: Tpnpy Were Evaluated Using Ptz-Induced Seizures
PMC12337321: Compound Not Identified
PMC12706759: Compound Not Identified
PMC12669438: Modulating Aqp4 Polarity
PMC12583660: Compound Not Identified
PMC12533203: Compound Not Identified
PMC12462991: Compound Not Identified
PMC12406720: Compound Not Identified
--------------------------------------


In [18]:
# ==========================================
# CÉLULA 8: Extração das Doses Experimentais
# ==========================================
# Objetivo: Identificar as doses administradas em mg/kg, analisando as 
# seções em ordem de prioridade. Utiliza janelas de contexto (sentenças) 
# para ignorar automaticamente doses associadas à BLACKLIST (controles/fármacos padrão).
# ==========================================

print("[INFO] Iniciando a extração de doses (mg/kg)...")

import re

extracted_doses_count = 0

for pmcid, data in validated_articles.items():
    doses_found = set()
    
    # Organizando as seções na ordem de prioridade exigida
    sections_to_search = [
        data.get("methods", ""),
        " ".join(data.get("tables", [])),
        data.get("results", ""),
        " ".join(data.get("figures", []))
    ]
    
    for section_text in sections_to_search:
        if not section_text:
            continue
            
        # Quebra o texto da seção em sentenças usando pontuação básica como delimitador
        sentences = re.split(r'(?<=[.!?])\s+', section_text)
        
        for sentence in sentences:
            sentence_lower = sentence.lower()
            
            # JANELA CONTEXTUAL: Se a sentença fala de um fármaco padrão ou controle, pulamos
            if any(black_word in sentence_lower for black_word in BLACKLIST):
                continue
                
            # REGEX: Captura números (incluindo listas separadas por vírgula, 'and', 'or', '-') antes de mg/kg
            # Ex: "12.5, 25 and 50 mg/kg" ou "10-50 mg/kg"
            matches = re.finditer(r"((?:\d+(?:\.\d+)?\s*(?:,|and|or|-|to)\s*)*\d+(?:\.\d+)?)\s*mg/?kg", sentence_lower)
            
            for match in matches:
                dose_string = match.group(1)
                
                # Extrai apenas os valores numéricos individuais da string capturada
                numeric_values = re.findall(r"\d+(?:\.\d+)?", dose_string)
                for num_str in numeric_values:
                    try:
                        dose_float = float(num_str)
                        # Ignoramos doses absurdamente altas ou zero (filtros de sanidade biológica)
                        if 0 < dose_float <= 5000:
                            doses_found.add(dose_float)
                    except ValueError:
                        continue
                        
        # Se encontramos doses no Methods, não precisamos garimpar as legendas (evita duplicação)
        if doses_found:
            break
            
    # Salva as doses ordenadas no dicionário do artigo
    validated_articles[pmcid]["doses_mg_kg"] = sorted(list(doses_found))
    
    if doses_found:
        extracted_doses_count += 1

print(f"[SUCCESS] Extração de doses concluída!")
print(f" - Artigos com doses válidas identificadas: {extracted_doses_count} de {len(validated_articles)}.")

# Diagnóstico de amostra
print("\n--- AMOSTRA DE DOSES EXTRAÍDAS ---")
sample_doses = {k: v["doses_mg_kg"] for k, v in list(validated_articles.items())[:15]}
for p_id, doses in sample_doses.items():
    print(f"{p_id}: {doses if doses else 'Nenhuma dose experimental capturada'}")
print("----------------------------------")

[INFO] Iniciando a extração de doses (mg/kg)...
[SUCCESS] Extração de doses concluída!
 - Artigos com doses válidas identificadas: 220 de 269.

--- AMOSTRA DE DOSES EXTRAÍDAS ---
PMC13336032: [5.0, 20.0, 50.0]
PMC13149591: [1.0, 2.0, 10.0, 70.0, 100.0, 127.0]
PMC13058834: [0.05, 0.5, 5.0, 60.0]
PMC12990634: [0.4, 2.0, 10.0, 30.0, 75.0, 90.0]
PMC12971609: [10.0, 100.0, 150.0]
PMC12956243: Nenhuma dose experimental capturada
PMC12954166: [300.0]
PMC12858582: [50.0]
PMC12337321: [55.0]
PMC12706759: [15.0, 27.0, 40.0, 660.0]
PMC12669438: [1.0, 30.0, 60.0, 130.0]
PMC12583660: [5.0, 30.0, 127.0]
PMC12533203: Nenhuma dose experimental capturada
PMC12462991: Nenhuma dose experimental capturada
PMC12406720: [0.5, 0.8, 2.0, 35.0, 100.0]
----------------------------------


In [22]:
# ==========================================
# CÉLULA 9: Extração de Latências com Dispersão (Média ± SD/SEM)
# ==========================================
# Objetivo: Capturar a média e a medida de dispersão atrelada,
# essenciais para a confiabilidade de modelos preditivos in silico.
# Retorna listas de tuplas no formato: (Média, Dispersão).
# ==========================================

print("[INFO] Iniciando a extração de Latências com Dispersão (Média ± SD/SEM)...")

import re

# Ampliando o léxico de busca
kws_crise = KEYWORDS.get("primeira_crise", []) + ["onset", "latency", "myoclonic", "jerk", "hlte", "time to"]
kws_morte = KEYWORDS.get("morte", []) + ["mortality", "survival", "lethality", "death"]

extracted_latencies_count = 0

for pmcid, data in validated_articles.items():
    # Usando SET para evitar capturar a mesma tupla duplicada no texto
    latencias_crise = set()
    latencias_morte = set()
    
    text_blocks = []
    if data.get("results"):
        text_blocks.extend(re.split(r'(?<=[.!?])\s+', data["results"]))
    if data.get("tables"):
        text_blocks.extend(data["tables"])
        
    for block in text_blocks:
        block_lower = block.lower()
        
        is_crise = any(kw in block_lower for kw in kws_crise)
        is_morte = any(kw in block_lower for kw in kws_morte)
        
        if not (is_crise or is_morte):
            continue
            
        # ==========================================
        # BUSCA: Padrão Média ± Dispersão com ou sem unidade
        # Captura: (Media) ± (Dispersao) e opcionalmente a (Unidade)
        # ==========================================
        p_dispersao = re.finditer(r"(\d+(?:\.\d+)?)\s*(?:±|\+/-)\s*(\d+(?:\.\d+)?)\s*(s\b|sec\b|seconds?\b|min\b|minutes?\b)?", block_lower)
        
        for m in p_dispersao:
            try:
                media = float(m.group(1))
                dispersao = float(m.group(2))
                unidade = m.group(3)
                
                # Se a unidade for minutos, converte ambos para segundos
                if unidade and "min" in unidade:
                    media *= 60.0
                    dispersao *= 60.0
                
                # Filtro de sanidade: ignoramos médias muito baixas para evitar 
                # capturar variáveis fisiológicas (ex: Peso 25 ± 2 g)
                if 10.0 <= media <= 7200.0:
                    if is_crise: latencias_crise.add((media, dispersao))
                    if is_morte: latencias_morte.add((media, dispersao))
            except ValueError:
                pass

    # Salvando os dados ordenados pela Média
    data["latencias_crise_s"] = sorted(list(latencias_crise), key=lambda x: x[0])
    data["latencias_morte_s"] = sorted(list(latencias_morte), key=lambda x: x[0])
    
    if data["latencias_crise_s"] or data["latencias_morte_s"]:
        extracted_latencies_count += 1

print(f"[SUCCESS] Extração de Média e Dispersão concluída!")
print(f" - Artigos com tuplas extraídas: {extracted_latencies_count} de {len(validated_articles)}.")

# ==========================================
# DIAGNÓSTICO
# ==========================================
print("\n--- AMOSTRA DE LATÊNCIAS EXTRAÍDAS (Média ± Dispersão em Segundos) ---")
success_sample = {k: v for k, v in validated_articles.items() if v["latencias_crise_s"] or v["latencias_morte_s"]}
for p_id, info in list(success_sample.items())[:15]:
    crises = info["latencias_crise_s"]
    mortes = info["latencias_morte_s"]
    print(f"[{p_id}] Crise(s): {crises if crises else 'N/A'} | Morte(s): {mortes if mortes else 'N/A'}")
print("----------------------------------------------------------------------")

[INFO] Iniciando a extração de Latências com Dispersão (Média ± SD/SEM)...
[SUCCESS] Extração de Média e Dispersão concluída!
 - Artigos com tuplas extraídas: 61 de 269.

--- AMOSTRA DE LATÊNCIAS EXTRAÍDAS (Média ± Dispersão em Segundos) ---
[PMC12956243] Crise(s): [(79.5, 22.88), (102.5, 15.55), (107.5, 11.68), (111.5, 56.97), (117.8, 12.12), (118.8, 13.77), (118.8, 21.7), (121.3, 18.28), (121.5, 56.85), (125.5, 12.23), (133.0, 6.272), (133.3, 15.06), (135.3, 27.66), (137.3, 13.05), (140.3, 6.652), (146.0, 62.71), (153.5, 38.0), (157.5, 11.9), (163.3, 21.5), (163.5, 35.2), (163.8, 48.54), (173.0, 9.274), (176.5, 36.52), (176.8, 64.52), (177.5, 41.51), (178.8, 17.5), (180.5, 14.64), (187.3, 37.82), (188.3, 88.97), (189.5, 21.75), (190.0, 43.01), (199.0, 50.64), (205.3, 71.44), (206.5, 40.32), (218.8, 12.84), (221.8, 74.96), (222.0, 88.88), (231.3, 46.61), (244.8, 165.7), (261.0, 92.47), (356.3, 269.5), (383.0, 259.4), (396.0, 316.2), (503.5, 248.3), (581.8, 252.3), (611.3, 792.9), (643

In [27]:
# ==========================================
# CÉLULA 10 (VERSÃO FINAL): Extração Cirúrgica (Prosa + Tabelas)
# ==========================================
# Objetivo: Extrair o Seizure Score do composto teste nos resultados 
# e dentro de tabelas, filtrando controles e drogas padrão por frase/linha.
# ==========================================

print("[INFO] Iniciando a extração cirúrgica da Escala de Racine (Resultados + Tabelas)...")

import re

# Padrões rigorosos de busca numérica
SCORE_PATTERNS = [
    r"(?:score|stage|grade|seizure)\s*(?:of|was|is|:)?\s*([0-6])\b",
    r"(?:racine)\s*(?:scale|stage|score)?\s*([0-6])\b"
]

RACINE_CONTEXT = ["racine", "score", "stage", "grade", "severity", "intensity"]

EXCLUDE_WORDS = [
    "control", "vehicle", "saline", "tween", "ptz alone", "untreated", 
    "diazepam", "valproate", "phenytoin", "phenobarbital", "reference", 
    "defined as", "scored as", "classified as", "0=none" 
]

extracted_scores_count = 0

for pmcid, data in validated_articles.items():
    racine_scores = set()
    
    # -----------------------------------------
    # 1. BUSCA NA PROSA (Resultados)
    # -----------------------------------------
    if data.get("results"):
        sentences = re.split(r'(?<=[.!?])\s+', data["results"])
        for sentence in sentences:
            sentence_lower = sentence.lower()
            
            if not any(ctx in sentence_lower for ctx in RACINE_CONTEXT):
                continue
            if any(ex_word in sentence_lower for ex_word in EXCLUDE_WORDS):
                continue
                
            for pattern in SCORE_PATTERNS:
                matches = re.finditer(pattern, sentence_lower)
                for m in matches:
                    try:
                        score_val = int(m.group(1))
                        if 0 <= score_val <= 6:
                            racine_scores.add(score_val)
                    except ValueError:
                        pass

    # -----------------------------------------
    # 2. BUSCA EM TABELAS (Linha por Linha)
    # -----------------------------------------
    if data.get("tables"):
        for table_text in data["tables"]:
            table_lower = table_text.lower()
            
            # Só analisa a tabela se ela for sobre severidade de crises
            if not any(ctx in table_lower for ctx in RACINE_CONTEXT):
                continue
            
            # Divide a tabela em linhas (assumindo que o parser manteve o \n)
            lines = table_lower.split('\n')
            for line in lines:
                # Se a linha for do controle ou diazepam, ignora a linha inteira!
                if any(ex_word in line for ex_word in EXCLUDE_WORDS):
                    continue
                
                # Procura os padrões nas linhas que restaram
                for pattern in SCORE_PATTERNS:
                    matches = re.finditer(pattern, line)
                    for m in matches:
                        try:
                            score_val = int(m.group(1))
                            if 0 <= score_val <= 6:
                                racine_scores.add(score_val)
                        except ValueError:
                            pass
                            
    data["escala_racine_teste"] = sorted(list(racine_scores))
    
    if data["escala_racine_teste"]:
        extracted_scores_count += 1

print(f"[SUCCESS] Extração Cirúrgica concluída!")
print(f" - Artigos com Seizure Score do Composto Teste extraídos: {extracted_scores_count} de {len(validated_articles)}.")

# ==========================================
# DIAGNÓSTICO DA CÉLULA 10
# ==========================================
print("\n--- AMOSTRA DE SCORES DE RACINE VALIDADOS (APENAS COMPOSTO TESTE) ---")
success_scores = {k: v for k, v in validated_articles.items() if v.get("escala_racine_teste")}

if not success_scores:
    print("[AVISO] Nenhum score foi encontrado após os filtros rígidos.")
else:
    for p_id, info in list(success_scores.items())[:15]:
        print(f"[{p_id}] Score(s) do composto teste filtrado(s): {info['escala_racine_teste']}")
print("---------------------------------------------------------------------")

[INFO] Iniciando a extração cirúrgica da Escala de Racine (Resultados + Tabelas)...
[SUCCESS] Extração Cirúrgica concluída!
 - Artigos com Seizure Score do Composto Teste extraídos: 40 de 269.

--- AMOSTRA DE SCORES DE RACINE VALIDADOS (APENAS COMPOSTO TESTE) ---
[PMC12956243] Score(s) do composto teste filtrado(s): [6]
[PMC12954166] Score(s) do composto teste filtrado(s): [4, 5]
[PMC12394891] Score(s) do composto teste filtrado(s): [3]
[PMC12260126] Score(s) do composto teste filtrado(s): [2, 4]
[PMC11698314] Score(s) do composto teste filtrado(s): [1, 2, 3]
[PMC11633471] Score(s) do composto teste filtrado(s): [4, 6]
[PMC10040380] Score(s) do composto teste filtrado(s): [1]
[PMC9617616] Score(s) do composto teste filtrado(s): [6]
[PMC9421656] Score(s) do composto teste filtrado(s): [6]
[PMC8873174] Score(s) do composto teste filtrado(s): [5]
[PMC8531497] Score(s) do composto teste filtrado(s): [3]
[PMC8517222] Score(s) do composto teste filtrado(s): [0, 3]
[PMC7957017] Score(s) do co

In [28]:
# ==========================================
# CÉLULA 11: Definição da Variável Target (Efeito Protetor Sim/Não)
# ==========================================
# Objetivo: Classificar o composto como "Sim" (Protetor) ou "Não" (Ineficaz).
# Lógica Híbrida: Cruza Análise Semântica (Conclusão/Discussão) com 
# o limite numérico da Escala de Racine extraída.
# ==========================================

print("[INFO] Iniciando a classificação Híbrida da Variável Target...")

import re

count_sim = 0
count_nao = 0

for pmcid, data in validated_articles.items():
    target = "Não" # Assumimos a hipótese nula (falha) como padrão
    
    # -----------------------------------------
    # 1. PREPARAÇÃO DO TEXTO (Conclusão + Final da Discussão)
    # -----------------------------------------
    text_blocks = []
    
    if data.get("conclusions"):
        text_blocks.append(data["conclusions"])
        
    if data.get("discussion"):
        # Pega aproximadamente os últimos 1500 caracteres (onde costuma estar a conclusão final)
        text_blocks.append(data["discussion"][-1500:])
        
    # Fallback: Se não houver conclusão nem discussão estruturada, usa o Abstract
    if not text_blocks and data.get("abstract"):
        text_blocks.append(data["abstract"])
        
    full_text = " ".join(text_blocks).lower()
    sentences = re.split(r'(?<=[.!?])\s+', full_text)
    
    text_positive = False
    text_negative = False
    
    # -----------------------------------------
    # 2. ANÁLISE SEMÂNTICA POR FRASE
    # -----------------------------------------
    for sentence in sentences:
        # Padrão de Falha (Explícita)
        if re.search(r"(did not|no significant|fail|ineffective|without effect).*(alter|change|protect|prevent|delay|prolong).*(seizure|latency|onset)", sentence):
            text_negative = True
            
        # Padrão de Aumento Estatístico da Latência
        if re.search(r"(significant|statistical).*(increas|prolong|delay).*(latency|onset|time to)", sentence):
            text_positive = True
            
        # Padrão de Proteção/Atenuação Geral
        if re.search(r"(protect|prevent|attenuat|anticonvulsant|ameliorat|abolish)", sentence) and re.search(r"(seizure|convulsion|epilep|mortality)", sentence):
            text_positive = True

    # -----------------------------------------
    # 3. ANÁLISE NUMÉRICA (Escala de Racine)
    # -----------------------------------------
    has_low_racine = False
    racine_scores = data.get("escala_racine_teste", [])
    
    if racine_scores:
        # Se o score mínimo atingido pelo composto foi <= 3, é um forte indício de proteção parcial ou total.
        if min(racine_scores) <= 3:
            has_low_racine = True

    # -----------------------------------------
    # 4. ÁRVORE DE DECISÃO FINAL
    # -----------------------------------------
    # Regra 1: Se o texto apontou sucesso e não há declaração explícita de falha
    if text_positive and not text_negative:
        target = "Sim"
        
    # Regra 2: Se os números de Racine provam proteção (score <= 3), o número vence um texto ambíguo
    if has_low_racine:
        target = "Sim"
        
    # Regra 3: Se o texto diz que falhou explicitamente e Racine não provou o contrário
    if text_negative and not has_low_racine:
        target = "Não"
        
    # Salva no dicionário
    data["target"] = target
    
    if target == "Sim":
        count_sim += 1
    else:
        count_nao += 1

print(f"[SUCCESS] Classificação da Variável Target concluída!")
print(f" - Compostos Protetores (Sim): {count_sim}")
print(f" - Compostos Ineficazes (Não): {count_nao}")

# ==========================================
# DIAGNÓSTICO
# ==========================================
print("\n--- AMOSTRA DE CLASSIFICAÇÃO TARGET ---")
sample_articles = list(validated_articles.items())[:15]
for p_id, info in sample_articles:
    rac_str = info.get('escala_racine_teste', 'N/A')
    print(f"[{p_id}] Racine Extraído: {rac_str} | Target Final: {info['target']}")
print("---------------------------------------")

[INFO] Iniciando a classificação Híbrida da Variável Target...
[SUCCESS] Classificação da Variável Target concluída!
 - Compostos Protetores (Sim): 119
 - Compostos Ineficazes (Não): 150

--- AMOSTRA DE CLASSIFICAÇÃO TARGET ---
[PMC13336032] Racine Extraído: [] | Target Final: Sim
[PMC13149591] Racine Extraído: [] | Target Final: Não
[PMC13058834] Racine Extraído: [] | Target Final: Não
[PMC12990634] Racine Extraído: [] | Target Final: Não
[PMC12971609] Racine Extraído: [] | Target Final: Não
[PMC12956243] Racine Extraído: [6] | Target Final: Não
[PMC12954166] Racine Extraído: [4, 5] | Target Final: Não
[PMC12858582] Racine Extraído: [] | Target Final: Não
[PMC12337321] Racine Extraído: [] | Target Final: Não
[PMC12706759] Racine Extraído: [] | Target Final: Sim
[PMC12669438] Racine Extraído: [] | Target Final: Não
[PMC12583660] Racine Extraído: [] | Target Final: Sim
[PMC12533203] Racine Extraído: [] | Target Final: Não
[PMC12462991] Racine Extraído: [] | Target Final: Não
[PMC1240672

In [29]:
# ==========================================
# CÉLULA 12: Consolidação e Exportação Dupla (CSV e XLSX)
# ==========================================
# Objetivo: Transformar o dicionário de dados minerados em um 
# DataFrame tabular (Pandas) estruturado para Machine Learning,
# e exportá-lo nos formatos CSV e Excel.
# ==========================================

print("[INFO] Convertendo dados brutos em DataFrame Pandas...")

import pandas as pd

rows = []

for pmcid, data in validated_articles.items():
    # Estruturando cada artigo como uma linha (registro) do dataset
    row = {
        "PMCID": pmcid,
        "Titulo": data.get("title", "Sem Título"),
        "Target_Efeito_Protetor": data.get("target", "Não"),
        "Doses_Testadas_mg_kg": str(data.get("doses", [])),
        "Latencias_Crises_M_SD_s": str(data.get("latencias_crise_s", [])),
        "Latencias_Morte_M_SD_s": str(data.get("latencias_morte_s", [])),
        "Score_Racine_Teste": str(data.get("escala_racine_teste", [])),
        # Extraindo um pequeno trecho da conclusão para fácil conferência humana no Excel
        "Trecho_Conclusao": data.get("conclusions", data.get("abstract", ""))[:300] + "..." 
    }
    rows.append(row)

# Criando o DataFrame principal
df_anticonvulsivantes = pd.DataFrame(rows)

print(f"[SUCCESS] DataFrame estruturado com sucesso!")
print(f" - Total de Linhas (Compostos/Artigos): {df_anticonvulsivantes.shape[0]}")
print(f" - Total de Colunas (Features): {df_anticonvulsivantes.shape[1]}")

# ==========================================
# DIAGNÓSTICO E EXIBIÇÃO
# ==========================================
print("\n--- AMOSTRA DO DATAFRAME FINAL ---")
try:
    display(df_anticonvulsivantes.head())
except NameError:
    print(df_anticonvulsivantes.head())
print("----------------------------------")

# ==========================================
# EXPORTAÇÃO PARA ARQUIVOS (CSV e EXCEL)
# ==========================================
nome_arquivo_csv = "dataset_ptz_anticonvulsivantes.csv"
nome_arquivo_excel = "dataset_ptz_anticonvulsivantes.xlsx"

# 1. Exportando como CSV (Separador ponto e vírgula para não bugar no Excel BR)
df_anticonvulsivantes.to_csv(nome_arquivo_csv, index=False, sep=";", encoding="utf-8")
print(f"\n[INFO] Arquivo CSV salvo com sucesso: '{nome_arquivo_csv}'")

# 2. Exportando como Excel (Requer openpyxl)
try:
    df_anticonvulsivantes.to_excel(nome_arquivo_excel, index=False, engine='openpyxl')
    print(f"[INFO] Arquivo Excel salvo com sucesso: '{nome_arquivo_excel}'")
except Exception as e:
    print(f"[ERRO] Falha ao salvar Excel. Verifique se a biblioteca openpyxl está instalada. Detalhe: {e}")

[INFO] Convertendo dados brutos em DataFrame Pandas...
[SUCCESS] DataFrame estruturado com sucesso!
 - Total de Linhas (Compostos/Artigos): 269
 - Total de Colunas (Features): 8

--- AMOSTRA DO DATAFRAME FINAL ---


,PMCID,Titulo,Target_Efeito_Protetor,Doses_Testadas_mg_kg,Latencias_Crises_M_SD_s,Latencias_Morte_M_SD_s,Score_Racine_Teste,Trecho_Conclusao
0,PMC13336032,Enhanced Glycolysis‐Driven Histone H3K18 Lacty...,Sim,[],[],[],[],ABSTRACT Metabolic reprogramming is increasing...
1,PMC13149591,"Design, synthesis, and in vivo antiepileptic e...",Não,[],[],[],[],Epilepsy impacts millions of individuals globa...
2,PMC13058834,"Fn14 is an activity-dependent, Bmal1-regulated...",Não,[],[],[],[],SUMMARY Cytokines and their receptors play imp...
3,PMC12990634,"Metabolism, pharmacokinetics, and anticonvulsa...",Não,[],[],[],[],"The imidazodiazepine, (5-(8-ethynyl-6-(pyridin..."
4,PMC12971609,HSDL2 Suppresses Epileptic Seizures Through Ph...,Não,[],[],[],[],ABSTRACT Background Temporal lobe epilepsy (TL...


----------------------------------

[INFO] Arquivo CSV salvo com sucesso: 'dataset_ptz_anticonvulsivantes.csv'
[INFO] Arquivo Excel salvo com sucesso: 'dataset_ptz_anticonvulsivantes.xlsx'
